# HPO SolarSystemLander

Elise HPO (8D) from `hpo/designs/design.md`.

## Setup

In [ ]:
# Colab setup
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys
from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.drive_backup import backup_to_drive, restore_from_drive
from hpo.lunar_lander.logging import configure_file_logging

drive.mount("/content/drive")
DRIVE_STUDY_DIR = Path("/content/drive/MyDrive/rl_lab/hpo")
LOCAL_STUDY_DIR = Path("/content/rl_lab/hpo/runs")
DRIVE_STUDY_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_STUDY_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
import torch

from dqn.vector_training import VectorTrainingConfig
from hpo.evaluation.dashboard import Dashboard
from hpo.objective import ObjectiveConfig
from hpo.solar_system_lander.environment import EnvFactory
from hpo.study import Baseline, StudyRunner

OBSERVATION_MODE = "8d"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OBJECTIVE_CFG = ObjectiveConfig(
    num_envs=20,
    device=device,
    eval_episodes=10,
)
ENVIRONMENT_FACTORY = EnvFactory(OBSERVATION_MODE)
RUN_NAME = "solar_system_lander_8d_elise"
STORAGE_PATH = LOCAL_STUDY_DIR / f"{RUN_NAME}.db"
DRIVE_STORAGE_PATH = DRIVE_STUDY_DIR / f"{RUN_NAME}.db"
LOG_PATH = LOCAL_STUDY_DIR / f"{RUN_NAME}.log"
DRIVE_LOG_PATH = DRIVE_STUDY_DIR / f"{RUN_NAME}.log"
restore_from_drive(DRIVE_STORAGE_PATH, STORAGE_PATH)
restore_from_drive(DRIVE_LOG_PATH, LOG_PATH)
configure_file_logging(LOCAL_STUDY_DIR, LOG_PATH.name)

def backup_study_to_drive():
    backup_to_drive(
        local_database=STORAGE_PATH,
        drive_database=DRIVE_STORAGE_PATH,
        local_log=LOG_PATH,
        drive_log=DRIVE_LOG_PATH,
    )

device, STORAGE_PATH, DRIVE_STORAGE_PATH


## Search spaces

In [ ]:
EPISODE_COUNTS = [500, 750, 1_000]

BASELINE_PARAMS = {
    "learning_rate": 0.0022727854024196057,
    "batch_size": 512,
    "eps_end": 0.047716002108220544,
    "eps_decay": 43_214,
    "gamma": 0.99,
    "tau": 0.005,
    "learning_starts": 2_500,
    "optimize_every": 2,
    "replay_memory_capacity": 400_000,
    "num_episodes": 500,
}

def vector_config(params):
    return VectorTrainingConfig(
        num_episodes=params["num_episodes"],
        batch_size=params["batch_size"],
        gamma=params["gamma"],
        eps_start=1.0,
        eps_end=params["eps_end"],
        eps_decay=params["eps_decay"],
        tau=params["tau"],
        learning_rate=params["learning_rate"],
        learning_starts=params["learning_starts"],
        optimize_every=params["optimize_every"],
    )

class SearchSpace1:
    def training_config(self, trial, params):
        return vector_config(params | {
            "num_episodes": trial.suggest_categorical(
                "num_episodes", EPISODE_COUNTS
            ),
        })

    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]

class SearchSpace2:
    def training_config(self, trial, params):
        return vector_config(params | {
            "eps_end": trial.suggest_float(
                "eps_end",
                0.03,
                0.07,
            ),
            "eps_decay": trial.suggest_int(
                "eps_decay",
                30_000,
                150_000,
                log=True,
            ),
        })

    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]


## Optimization

In [ ]:
runner = StudyRunner(
    database_path=lambda _name: STORAGE_PATH,
    environment_factory=ENVIRONMENT_FACTORY,
    objective_cfg=OBJECTIVE_CFG,
    baseline=Baseline(params=BASELINE_PARAMS, score=118.09342221642783),
    reporter=Dashboard(),
    study_attrs=ENVIRONMENT_FACTORY.metadata(),
    robust_candidates=5,
    extra_seeds=(1001, 1002, 1003, 1004),
    sync_fn=backup_study_to_drive,
)

runner.run("s1_flight_hours", SearchSpace1(), 25)
runner.run("s2_exploration", SearchSpace2(), 25)


## Reload studies after a runtime reset

In [ ]:
# Run only after the runtime environment has been reset.
import optuna

def load_study(name):
    return optuna.load_study(study_name=name, storage=f"sqlite:///{STORAGE_PATH}")

study1 = load_study("s1_flight_hours")
study2 = load_study("s2_exploration")

## Analysis

In [ ]:
import pandas as pd

studies = [study1, study2]
labels = ["S1", "S2"]

rows = []
for label, study in zip(labels, studies):
    rows.append({
        "study": label,
        "score": study.user_attrs["incumbent_score"],
    })

display(pd.DataFrame(rows).set_index("study").style.format("{:.1f}"))

### Is the lander still learning at the end?

In [ ]:
from pathlib import Path

import optuna
import pandas as pd
import plotly.express as px

DATABASE_NAME = "solar_system_lander_8d_elise.db"
STUDY_NAME = "s2_exploration"

tmp_dir = Path("hpo/tmp")
if not tmp_dir.exists():
    tmp_dir = Path("../tmp")
database_path = (tmp_dir / DATABASE_NAME).resolve()
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=f"sqlite:///{database_path}",
)
params = study.user_attrs.get("robust_best_params")
trials = [
    trial for trial in study.trials
    if trial.state.name == "COMPLETE" and trial.params == params
]
#trial = max(trials, key=lambda trial: trial.value) if trials else study.best_trial
trial = study.best_trial

returns = trial.user_attrs["training_curve"]["episode_returns"]
window = min(100, len(returns) // 2)
curve = pd.DataFrame({"Episode return": returns})
curve[f"Mean ({window} episodes)"] = curve["Episode return"].rolling(window).mean()

display(px.line(curve, labels={"index": "Episode", "value": "Return"}))

previous_mean = curve["Episode return"].iloc[-2 * window:-window].mean()
final_mean = curve["Episode return"].iloc[-window:].mean()
print(f"Previous {window} episodes: {previous_mean:.1f}")
print(f"Final {window} episodes:    {final_mean:.1f}")
print(f"Change:                    {final_mean - previous_mean:+.1f}")

Previous 100 episodes: -31.8
Final 100 episodes:    34.9
Change:                    +66.7


### Should the trials have trained longer?

In [20]:
import numpy as np

def estimate_alpha(returns, window):
    returns = np.asarray(returns)
    means = [
        returns[-3 * window:-2 * window].mean(),
        returns[-2 * window:-window].mean(),
        returns[-window:].mean(),
    ]
    previous_gain = means[1] - means[0]
    latest_gain = means[2] - means[1]
    return np.nan if np.isclose(previous_gain, 0) else latest_gain / previous_gain

In [24]:
import plotly.graph_objects as go

rows = []
for trial in study.trials:
    if trial.state.name != "COMPLETE" or "training_curve" not in trial.user_attrs:
        continue

    returns = np.asarray(trial.user_attrs["training_curve"]["episode_returns"])
    window = min(50, len(returns) // 3)
    previous = returns[-2 * window:-window]
    final = returns[-window:]
    improvement = final.mean() - previous.mean()
    error = 1.96 * np.sqrt(
        previous.var(ddof=1) / window + final.var(ddof=1) / window
    )
    alpha = estimate_alpha(returns, window)
    rows.append((
        trial.number,
        improvement,
        error,
        improvement > error,
        alpha,
    ))

figure = go.Figure(go.Bar(
    x=[row[0] for row in rows],
    y=[row[1] for row in rows],
    error_y={"type": "data", "array": [row[2] for row in rows]},
    marker_color=["green" if row[3] else "gray" for row in rows],
    text=[f"α={row[4]:.1f}" if np.isfinite(row[4]) else "α=n/a" for row in rows],
    textposition="outside",
    textfont={"size": 14},
))
figure.add_hline(y=0, line_color="black")
figure.update_layout(
    xaxis_title="Trial",
    yaxis_title="Change in mean episode return",
    title=(
        "Green: evidence that longer training may help"
        "<br><sup>Balken: Differenz der Mittelwerte der letzten zwei Fenster</sup>"
    ),
)
figure.show()

print("window = " + str(window))

window = 50
